# Federated learning with Opacus DP-SGD and OpenDP privacy accounting

A minimal end-to-end example. Three simulated clients train a small model with
FedAvg. Each client runs **DP-SGD with Opacus** on its own records, and each
client's privacy loss is tracked by an **OpenDP odometer**. Every local round is
executed by the odometer.

**Status:** a proposed example for review by the OpenDP maintainers. It has not
been reviewed, accepted or merged by OpenDP. It is a research demonstration on
synthetic data, simulated in one process, not a production DP deployment. See
section 9 and `PRIVACY_ASSUMPTIONS.md`.

**Why both libraries?** Opacus provides the mechanism: per-example gradients,
clipping and noise for PyTorch models. OpenDP provides a framework in which each
release is a `Measurement` with a stated privacy map, and composition is tracked
by vetted combinators.

| Responsibility | Done by |
|---|---|
| Per-example clipping and Gaussian noise (the mechanism) | Opacus `GradSampleModule` + `DPOptimizer` (`secure_mode=True`) |
| Poisson subsampling with a declared rate and step count | `CSPRNGPoissonSampler` (OS CSPRNG; same semantics as Opacus's `UniformWithReplacementSampler`, which is kept for debug mode) |
| Wrapping each local round as a measurement | OpenDP `make_user_measurement` (`renyi_divergence`) |
| Composing rounds per client and running each round | OpenDP `make_fully_adaptive_composition` odometer |
| Composing a prospective round for the budget check | OpenDP `make_composition` |
| RDP curve of the sampled Gaussian; RDP → (ε, δ) | this example (`src/accounting.py`), checked against Opacus, Google `dp_accounting` and numerical integration |
| FedAvg over clients | this example (`src/fl.py`) |

Opacus's `PrivacyEngine` accountant is **not** used for any reported number.

In [1]:
import platform, warnings
warnings.filterwarnings("ignore", message="Full backward hook")  # PyTorch notice: inputs do not require grad
import numpy as np, torch, opacus
from importlib.metadata import version
import opendp.prelude as dp

from src import accounting as acc
from src import fl

acc.enable_opendp_features()   # "contrib" + "honest-but-curious", required by make_user_measurement
print("python", platform.python_version(), "| torch", torch.__version__,
      "| opacus", opacus.__version__, "| opendp", version("opendp"), "| numpy", np.__version__)

python 3.13.16 | torch 2.14.1+cu130 | opacus 1.6.0 | opendp 0.16.0 | numpy 2.5.4


## 1. Privacy setting

* **Privacy unit:** one record (record-level DP), not one client.
* **Neighbouring datasets:** two local datasets of one client that differ by adding or
  removing one record. Both must be non-empty, which is the input domain. In
  OpenDP terms this is `symmetric_distance` with `d_in = 1`; other `d_in`
  values are rejected.
* **Adversary:** the coordinator and every other client. They see every model
  update a client sends, in every round. No secure aggregation is assumed, and
  none is needed for the guarantee.
* **Trusted:** each client's own training code and randomness.
* **Budget:** held by the client. Each client owns an odometer bound to its own
  data. A round that would exceed any client's budget is refused before any
  training runs.

## 2. Data and a disjoint, non-IID partition

Records are routed to clients by fixed, public thresholds on feature 0, which
depend only on the record itself. So the client datasets are disjoint, and adding or
removing a record never moves another record. That is what lets a record's privacy
loss be the loss of the one client that holds it. `make_clients` also refuses any
record id held by two clients.

The client sizes are printed below to show the reader the partition. They are
**not** part of any DP release, and the mechanism does not use them.

In [2]:
D = 8
X, y, X_test, y_test = fl.make_synthetic_data(n_train=3000, n_test=1000, d=D, seed=0)
parts = fl.partition_records(X)
client_sets = [fl.ClientDataset(X[p], y[p], k, record_ids=p) for k, p in enumerate(parts)]
for ds, p in zip(client_sets, parts):
    print(f"client {ds.client_id}: n={len(ds):4d}  mean(feature 0)={X[p,0].mean():+.2f}  positive rate={y[p].mean():.2f}"
          "   (illustration only, not released)")

client 0: n=1024  mean(feature 0)=-1.04  positive rate=0.64   (illustration only, not released)
client 1: n= 927  mean(feature 0)=-0.00  positive rate=0.48   (illustration only, not released)
client 2: n=1049  mean(feature 0)=+1.05  positive rate=0.38   (illustration only, not released)


## 3. DP-SGD configuration: public constants only

Every parameter of the mechanism is a declared public constant: q, σ, C, K and the
normaliser B (`expected_batch_size`). **None is computed from a client's data or
dataset size.** Poisson inclusion decisions use the OS CSPRNG with an exact integer
threshold, so each record's inclusion probability is ⌊q·2⁵³⌋/2⁵³ ≤ q. The accountant's
q is therefore an upper bound on the true rate. So two neighbouring datasets run exactly the same mechanism. Each step
releases (Σᵢ clip(gᵢ, C) + N(0, σ²C²)) / B, with an independent Bernoulli(q)
inclusion for every record. Dividing by a public B is post-processing.

An earlier draft used B = round(q · n_k). That made the update depend on the
client's size n_k, which the privacy map does not account for, so it was replaced by
a fixed B. Utility is best when B ≈ q · n_k, but privacy does not depend on it.

In [3]:
cfg = fl.DPConfig(sample_rate=0.05, noise_multiplier=1.0, max_grad_norm=1.0, expected_batch_size=50,
                  local_steps=20, rounds=5, lr=0.5, target_epsilon=5.0, delta=1e-5)
cfg

DPConfig(sample_rate=0.05, noise_multiplier=1.0, max_grad_norm=1.0, expected_batch_size=50, local_steps=20, rounds=5, lr=0.5, target_epsilon=5.0, delta=1e-05, secure_noise=True, sampling_rng='csprng', reproducible_noise=False)

## 4. One local round as an OpenDP measurement

`make_dp_sgd_round` wraps the Opacus training function as an OpenDP
`Measurement` whose output measure is `RenyiDivergence`. Its privacy map
returns the RDP curve of K steps of the Poisson-subsampled Gaussian
(Mironov, Talwar & Zhang 2019). OpenDP's `make_composition` adds these curves.

In [4]:
dom = fl.client_domain()
m = acc.make_dp_sgd_round(dom, function=lambda data: None, sample_rate=cfg.sample_rate,
                          noise_multiplier=cfg.noise_multiplier, steps=cfg.local_steps)
print(m.input_metric, "->", m.output_measure)
curve = m.map(1)                                   # d_in = 1 record
print("RDP of one round at alpha = 2, 8, 32:", [round(curve(a), 4) for a in (2.0, 8.0, 32.0)])
two = dp.c.make_composition([m, m]).map(1)         # OpenDP composes RDP curves additively
print("two rounds composed by OpenDP at alpha = 8:", round(two(8.0), 4), "= 2 x", round(curve(8.0), 4))

SymmetricDistance() -> RenyiDivergence
RDP of one round at alpha = 2, 8, 32: [0.0857, 12.0254, 258.1526]
two rounds composed by OpenDP at alpha = 8: 24.0508 = 2 x 12.0254


## 5. Federated training

Each round, every client's budget is checked first, so a round runs for all
clients or for none. The check evaluates each privacy map *before* any training. In
OpenDP 0.16.0 the odometer itself would run a measurement whose map is broken, and
fail only later. A round that fails mid-training is still charged, and that client is
halted. Then each client builds a measurement around the current
global model and submits it to **its own odometer**, which runs the Opacus
training and records the round. Only the model state is released. The
coordinator averages the models with uniform, data-independent weights, which is
post-processing.

In [5]:
clients = fl.make_clients(client_sets, cfg)
init_state = fl.get_state(fl.make_model(D, seed=0))
print("before training:", {k: round(v, 3) for k, v in fl.evaluate(init_state, X_test, y_test, D).items()})
result = fl.run_federated(clients, init_state, cfg, D, X_test=X_test, y_test=y_test)
print(f"{'round':>5} {'accuracy':>9} {'loss':>7} " + " ".join(f"eps_c{c.data.client_id:<4}" for c in clients))
for row in result["history"]:
    print(f"{row['round']:>5} {row['accuracy']:>9.3f} {row['loss']:>7.3f} "
          + " ".join(f"{row[f'eps_client_{c.data.client_id}']:<8.4f}" for c in clients))

before training: {'accuracy': 0.447, 'loss': 0.744}


round  accuracy    loss eps_c0    eps_c1    eps_c2   
    1     0.726   0.598 2.5499   2.5499   2.5499  
    2     0.787   0.484 2.9963   2.9963   2.9963  
    3     0.792   0.471 3.3681   3.3681   3.3681  
    4     0.794   0.467 3.7399   3.7399   3.7399  
    5     0.791   0.477 4.1117   4.1117   4.1117  


## 6. How ε is calculated, and what it means

1. Per step: the RDP of the Poisson-subsampled Gaussian with rate q and noise
   multiplier σ, at integer orders α = 2…256.
2. Per round: K × that curve (privacy map of the measurement).
3. Per client: the OpenDP odometer composes the rounds (sum of curves).
4. (ε, δ): ε = min over α of rdp(α) + log((α−1)/α) − (log δ + log α)/(α−1)
   (Canonne, Kamath & Steinke 2020; Balle et al. 2020).
5. Federation: the client datasets are disjoint, and a client's mechanism is the only
   one that reads its records. So the record-level guarantee is the **maximum over
   clients** (parallel composition, assumption A2). For contrast, OpenDP's
   sequential composition of *all* rounds of *all* clients is the bound needed
   if a record could appear at several clients.

ε depends only on (q, σ, K, rounds, δ). It does not depend on B or n_k. The PLD
number printed below is **not** the guarantee of this code. It shows how much slack
the RDP route leaves, which is one reason native PRV/PLD support in OpenDP would help.

In [6]:
from opacus.accountants.analysis.rdp import compute_rdp, get_privacy_spent
per_client = {c.data.client_id: c.odometer.epsilon_spent() for c in clients}
for k, (eps, order) in per_client.items():
    print(f"client {k}: epsilon = {eps:.4f} at delta = {cfg.delta} (best Renyi order {order})")
eps_parallel = max(e for e, _ in per_client.values())
eps_sequential = acc.sequential_bound([m for c in clients for m in c.odometer.measurements], cfg.delta)
print(f"\nrecord-level epsilon, disjoint clients (max):        {eps_parallel:.4f}")
print(f"record-level epsilon if records could repeat (sum): {eps_sequential:.4f}")

# Independent cross-checks only (not used for the guarantee)
orders = list(range(2, acc.MAX_ORDER + 1))
T = cfg.local_steps * cfg.rounds
ref = get_privacy_spent(orders=orders, delta=cfg.delta,
                        rdp=compute_rdp(q=cfg.sample_rate, noise_multiplier=cfg.noise_multiplier, steps=T, orders=orders))
print(f"cross-check, Opacus RDP analysis for {T} steps:         epsilon = {float(ref[0]):.4f}")
import dp_accounting as dpa
event = dpa.SelfComposedDpEvent(dpa.PoissonSampledDpEvent(cfg.sample_rate, dpa.GaussianDpEvent(cfg.noise_multiplier)), T)
g_rdp = dpa.rdp.RdpAccountant(orders=[float(a) for a in orders]); g_rdp.compose(event)
g_pld = dpa.pld.PLDAccountant(); g_pld.compose(event)
print(f"cross-check, Google dp_accounting RDP (same orders):  epsilon = {g_rdp.get_epsilon(cfg.delta):.4f}")
print(f"for reference, Google dp_accounting PLD (tighter):    epsilon = {g_pld.get_epsilon(cfg.delta):.4f}")
assert abs(float(ref[0]) - eps_parallel) < 1e-9 and abs(g_rdp.get_epsilon(cfg.delta) - eps_parallel) < 1e-6

client 0: epsilon = 4.1117 at delta = 1e-05 (best Renyi order 5)
client 1: epsilon = 4.1117 at delta = 1e-05 (best Renyi order 5)
client 2: epsilon = 4.1117 at delta = 1e-05 (best Renyi order 5)



record-level epsilon, disjoint clients (max):        4.1117
record-level epsilon if records could repeat (sum): 6.5127
cross-check, Opacus RDP analysis for 100 steps:         epsilon = 4.1117


cross-check, Google dp_accounting RDP (same orders):  epsilon = 4.1117
for reference, Google dp_accounting PLD (tighter):    epsilon = 3.5021


## 7. Budget enforcement

Each client's target is ε = 5 at δ = 1e-5. Keep requesting rounds: the first
round that would exceed any client's budget is refused **before** any training.
The decision depends only on public parameters.

In [7]:
state, extra = result["state"], 0
try:
    while True:
        state = fl.run_federated(clients, state, cfg, D, rounds=1)["state"]
        extra += 1
except acc.BudgetExceeded as e:
    print(f"ran {extra} extra round(s), then refused: {e}")
print("rounds recorded per client:", [c.odometer.rounds_run for c in clients])
print("final epsilon per client:  ", [round(c.odometer.epsilon_spent()[0], 4) for c in clients])

ran 3 extra round(s), then refused: round 9 would raise epsilon to 5.1428 > target 5.0 at delta=1e-05
rounds recorded per client: [8, 8, 8]


final epsilon per client:   [4.9145, 4.9145, 4.9145]


## 8. What OpenDP does natively here, and the gaps

| Piece | OpenDP 0.16.0 | Here |
|---|---|---|
| Round as a measurement over `RenyiDivergence` | ✅ `make_user_measurement` (needs `honest-but-curious`) | used |
| Composing RDP curves | ✅ `make_composition`, `make_fully_adaptive_composition` | used |
| RDP of the Poisson-subsampled Gaussian | ❌ not provided | `sampled_gaussian_rdp`; agrees with Opacus's analysis (rel. 1e-9) and with quadrature of the Rényi divergence |
| RDP → (ε, δ) measure cast | ❌ none found | `rdp_to_approx_dp` |
| Odometer with a broken privacy map | ⚠️ runs the function and releases its output, then raises only when `privacy_loss` is queried | map evaluated before running; fail closed |
| Privacy filter over an RDP odometer | ⚠️ `make_privacy_filter(odometer, d_in=1, d_out=curve)` aborted the interpreter (Rust panic `unrecognized generic "Function"`), with `curve` as a Python callable and as an OpenDP `Function`. It may be unsupported rather than a bug | Python budget check using OpenDP `make_composition` |
| Parallel composition over disjoint clients | ❌ not expressed in OpenDP here | max over clients, under assumption A2 |
| Opacus DP-SGD as a native OpenDP mechanism | ⏳ upcoming OpenDP–Opacus branch | Opacus called inside the user measurement |

**When the OpenDP–Opacus branch lands**, the intended change is to replace
`make_dp_sgd_round` and `sampled_gaussian_rdp` with the native constructor. The odometer,
the budget check and the FL loop stay. That removes the need for `honest-but-curious`
in the mechanism itself.

## 9. Limitations (read before reusing)

* **Research demonstration.** Synthetic data, simulated clients, one process. There
  is none of the operational isolation of a real deployment: network, separate
  machines, attestation.
* **Noise randomness is not cryptographic.** Poisson sampling uses the OS CSPRNG.
  Gaussian noise uses PyTorch's Mersenne Twister, freshly seeded per client and round
  from `secrets.randbits`, with Opacus `secure_mode=True`: the floating-point
  mitigation of Holohan & Braghin (2021), four Gaussian draws per coordinate.
  `torchcsprng` has no distribution for this Python/PyTorch, so a CSPRNG-backed noise
  generator could not be used. The accounting assumes ideal Gaussian noise.
* **`reproducible_noise=True`** seeds the noise with known values for debugging. It
  voids the guarantee and warns.
* **Public metadata:** q, σ, C, B, K, the number of rounds, δ, the architecture
  and the learning rate. The hyperparameters were chosen by hand on synthetic
  data, not privately.
* **Trust in the client code.** The guarantee relies on the privacy map correctly
  describing the training (the `honest-but-curious` contract). The tests check
  this empirically and against independent references, but do not prove it.